####  <font color='red'>任务二（二分类问题）：基于胸部X光图像的自动分类</font>
**任务目标：**

针对胸部X光图像，
1. 对这个数据集上的工作进行调研（a. 调研所有文章，给出方法分类/演进脉络；或者，b. 对至少三篇文章，进行详细的方法演变轨迹分析）；
2. 设计并实现一个深度学习模型，用于自动分类正常肺部影像与肺炎影像（a, 自己构造，需要做出两个方案，并进行比较；或者，b. 先复现已有工作，自己加入一些改动，分析为什么性能变好还是变差）；
3. 分析中间特征（查看目前工作中，用于中间过程分析/模型可解释性的方式有哪些）。

**数据集：**
数据集包含来自多个来源的胸部X光影像，图像数据为JPEG格式，确保图像清晰度和可识别性，以便于后续的处理和模型训练。数据集具体包含：
* **正常胸部X光影像:** 这些图像展示健康肺部的正常解剖结构，用于训练模型识别无病变的影像特征。

* **肺炎胸部X光影像:** 这些图像展示因肺炎引起的病变，如浸润影、模糊边界等，提供模型学习病变特征的依据。

**处理流程参考：**

* **数据集调研：** 查找这个数据集是哪一年出来的，目前刷榜的情况如何，方法有哪些类别，这些方法专注的问题有哪些？
* **数据预处理：** 加载图像数据，进行标准化处理，包括调整图像尺寸、灰度归一化和数据增强（如旋转、翻转、平移等），提高模型的训练稳定性和鲁棒性。
* **模型设计及训练：** 构建能够对正常和肺炎图像进行分类的深度学习模型，如ResNet或Unet，选择70%的数据作为训练集进行模型训练。利用数据增强技术提升模型的泛化能力，并使用早停法防止过拟合。
* **模型评估：** 使用交叉验证和测试集（剩余30%数据）评估模型性能，计算准确率、灵敏度、特异度及AUC等指标，综合评估模型的分类能力，并绘制ROC曲线，展示模型的分类效果。
* **结果可视化：** 提供分类结果的可视化，如混淆矩阵、ROC曲线等，生成报告以帮助医生理解模型在胸部X光影像分类任务中的表现和决策依据。
* **模型可解释性：** 将网络中间层特征（最后一层，倒数第二层等），用TSNE进行降维，观察不同类别的特征分布；或者用CradCAM技术对模型关注区域进行分析。

####  <font color='red'>任务三（生成问题）：基于RetinaMNIST数据集的扩散模型图像生成</font>
**任务目标：**

针对RetinaMNIST数据集，设计并实现一个基于扩散模型（Diffusion Model）的生成模型，用于自动生成与真实样本分布相似的医学影像。该任务旨在探索扩散模型在二维医学影像生成中的应用与性能表现。

**数据集：**
数据集采用RetinaMNIST（[https://medmnist.com/](https://medmnist.com/)），包含经过标准化处理的视网膜影像数据。  
* **图像尺寸：** 28×28像素，灰度或RGB格式。  
* **任务性质：** 将原分类任务转化为生成任务，通过学习真实图像分布生成新的样本，用于验证模型的生成能力。  
* **数据格式：** `.npz`或`.png`文件，可通过MedMNIST官方接口直接加载。

**处理流程参考：**

* **数据预处理：** 下载并加载RetinaMNIST数据，调整图像尺寸为28×28，进行归一化处理（如映射至[–1, 1]区间），并划分训练集与验证集。

* **模型设计及训练：** 实现标准去噪扩散概率模型（Denoising Diffusion Probabilistic Model, DDPM），包括前向加噪过程与反向去噪生成过程。模型输入为28×28图像，输出同尺寸重建结果。训练过程中记录损失曲线变化，并在不同训练阶段保存生成样本以观察生成质量变化。

* **模型评估：** 使用训练集和验证集分别计算并分析模型性能，采用以下评估方式：
  - **原图/生成图对比：** 展示真实图像与模型生成图像的可视化结果，对比图像结构与质量；
  - **FID（Fréchet Inception Distance）：** 衡量生成样本与真实样本在特征空间分布上的差异；
  - **SSIM（Structural Similarity Index）：** 衡量生成图像与真实图像在亮度、对比度和结构等方面的相似程度；
  - **训练曲线记录：** 展示模型训练与验证集的在损失和FID,SSIM变化趋势，分析模型的收敛性与稳定性。

* **结果可视化：** 提供生成结果的可视化和评估报告，包括：
  - 真实图像与生成图像的对比示例；
  - 不同训练阶段的生成样本对比；
  - FID与SSIM得分；
  - 模型训练损失曲线。


### <font color='red'>示例</font>
下面我们给出任务一的示例代码，用于完成脑瘤分割网络的构建和浸润区域不确定性的估计。

#### 数据预处理
1、原数据集BraTS共包含T1n、T1c、T2f、T2w、seg五类3D图像数据，不同数据适合处理不同目标任务。此处以T1c数据作为演示示例，首先将nii.gz数据转化为2D图像数据，T1c和seg转化结果如下图所示。其中，seg image：深灰色—坏死区域；白色—肿瘤强化区域；浅灰色—瘤周水肿区域（浸润区域）。
<img src="./t1c.png" width="150" style="display:inline-block;">
<img src="./seg.png" width="150" style="display:inline-block;">

2、为了实现4种类别（背景、坏死、增强、浸润）的分割任务，对seg image进行标准化处理。首先，遍历全部图像像素值，seg image共包含 [0, 85, 128, 170, 255] 5类像素点。

基于先前的工作已知对应关系"0:背景"、"85:坏死"、"170:浸润"、"255:增强"，128像素点为未知标注。因此，我们将seg image转化为mask标准形式，并删除包含128像素图像。

In [ ]:
import os
from PIL import Image
import numpy as np

In [ ]:
# [0, 85, 170, 255] --> [0, 1, 3, 2]
mask_dir = r''

for mask_name in os.listdir(mask_dir):
    mask_path = os.path.join(mask_dir, mask_name)
    data = Image.open(mask_path).convert("L")

    w, h = data.size

    # 遍历所有像素点
    for i in range(h):
        for j in range(w):
            pixel_value = data.getpixel((j, i))
            if pixel_value == 85:
                data.putpixel((j, i), 1)
            elif pixel_value == 255:
                data.putpixel((j, i), 2)
            elif pixel_value == 170:
                data.putpixel((j, i), 3)

    data.save(os.path.join(r'', mask_name.split(".")[0] + '.png'))

In [ ]:
# Delete 128
# 使用集合来存储唯一的像素值
unique_pixel_values = set()

for name in os.listdir(r''):
    image_path = os.path.join(r'', name)
    image = Image.open(image_path)
    image_array = np.array(image)
    pixel_values = image_array.flatten()
    
    if 128 in pixel_values:
        print(name)
        os.remove(image_path)

3、为了提高分割网络的收敛速度、精简数据集，遍历全部mask图像，如果像素值全为0，则删除该图像。

In [ ]:
# Delete all 0 pixels
mask_dir = r''

# 遍历目录中的每个图像文件
for filename in os.listdir(mask_dir):
    image_path = os.path.join(mask_dir, filename)

    # 打开图像并转换为numpy数组
    image = Image.open(image_path)
    image_array = np.array(image)

    # 检查图像是否全黑
    if np.all(image_array == 0):
        print(f'Deleting black image: {filename}')
        os.remove(image_path)  # 删除全黑的图像

#### 模型设计
我们采用Unet 2D分割网络，将网络的输出设为4，为了实现浸润区域的不确定估计，我们在Unet的基础上添加MC Dropout模块，用于在每次训练时随机丢弃一部分神经元，从而增强模型的鲁棒性，并提供不确定性估计。

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

In [ ]:
# 构建网络框架 Unet + MC dropout
class UNet(nn.Module):
    def __init__(self, in_channels, out_channels, p_dropout=0.5):
        super(UNet, self).__init__()
        self.p_dropout = p_dropout

        # 编码器部分
        self.encoder1 = self.conv_block(in_channels, 64)
        self.encoder2 = self.conv_block(64, 128)
        self.encoder3 = self.conv_block(128, 256)
        self.encoder4 = self.conv_block(256, 512)
        self.bottleneck = self.conv_block(512, 1024)

        # 解码器部分
        self.decoder4 = self.upconv_block(1024, 512)
        self.decoder4_conv = self.conv_block(1024, 512, apply_dropout=True)
        self.decoder3 = self.upconv_block(512, 256)
        self.decoder3_conv = self.conv_block(512, 256, apply_dropout=True)
        self.decoder2 = self.upconv_block(256, 128)
        self.decoder2_conv = self.conv_block(256, 128, apply_dropout=True)
        self.decoder1 = self.upconv_block(128, 64)
        self.decoder1_conv = self.conv_block(128, 64, apply_dropout=True)

        # 最终输出卷积层
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def conv_block(self, in_channels, out_channels, apply_dropout=False):
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        ]
        if apply_dropout:
            layers.append(nn.Dropout(p=self.p_dropout))
        return nn.Sequential(*layers)

    def upconv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(F.max_pool2d(enc1, kernel_size=2))
        enc3 = self.encoder3(F.max_pool2d(enc2, kernel_size=2))
        enc4 = self.encoder4(F.max_pool2d(enc3, kernel_size=2))

        bottleneck = self.bottleneck(F.max_pool2d(enc4, kernel_size=2))

        dec4 = self.decoder4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.decoder4_conv(dec4)

        dec3 = self.decoder3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.decoder3_conv(dec3)

        dec2 = self.decoder2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2_conv(dec2)

        dec1 = self.decoder1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1_conv(dec1)

        return self.final_conv(dec1)

In [ ]:
# 自定义数据集类
class BraTSDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = os.listdir(image_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = os.path.join(self.image_dir, self.images[idx])
        mask_name = os.path.join(self.mask_dir, self.images[idx])
        image = Image.open(img_name).convert("L")
        mask = Image.open(mask_name).convert("L")

        if self.transform:
            image = self.transform(image)
            mask = torch.from_numpy(np.array(mask)).long()

        return image, mask

# 数据转换
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

image_dir = './image'
mask_dir = './mask'
dataset = BraTSDataset(image_dir, mask_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=4)

#### 模型训练

In [ ]:
# 初始化模型
in_channels = 1
out_channels = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(in_channels, out_channels, p_dropout=0.5).to(device)
# 损失和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
# 模型训练
num_epochs = 25
model_save_dir = './model_checkpoints'
os.makedirs(model_save_dir, exist_ok=True)

for epoch in range(num_epochs):
    model.train()
    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')
    
    # 保存模型
    model_save_path = os.path.join(model_save_dir, f'unet_model_epoch_{epoch + 1}.pth')
    torch.save(model.state_dict(), model_save_path)
    print(f'Model parameters saved to {model_save_path}')

#### 模型验证
1、完成训练后，我们加载训练得到的权重，通过验证实现脑瘤、增强区域的分割。

In [ ]:
# 加载模型
in_channels = 1
out_channels = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(in_channels, out_channels, p_dropout=0.5).to(device)
model.load_state_dict(torch.load("./model_checkpoints/unet_model_epoch_25.pth", map_location=device))
model.eval()  # 评估模式

# 定义加载和处理图像的函数
def load_and_preprocess_image(image_path):
    image = Image.open(image_path).convert("L")  # 将图像转换为灰度图
    input_tensor = transform(image).unsqueeze(0).to(device)  # 添加 batch 维度
    return input_tensor, image

# 定义数据转换
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# 定义可视化函数
def visualize_and_save_outputs(image, outputs, save_dir, base_name):
    num_classes = outputs.shape[1]  # 输出类别数量 (batch_size, num_classes, H, W)

    # 应用 softmax 以获取每个类别的概率
    outputs = torch.softmax(outputs, dim=1)  # 对类别维度应用 softmax

    for i in range(num_classes -1): # 由于我们不进行浸润区域的分割，这里的类别减一
        output = outputs[0, i].cpu().detach().numpy()  # 取第一张图像的第 i 类输出

        # 二值化处理
        threshold = 0.5  # 可以调整阈值
        output_binary = (output > threshold).astype(np.uint8)
        # 将二值化图像转换为可视化格式（黑白）
        output_image = Image.fromarray(output_binary * 255)
        # 保存图像
        output_image.save(os.path.join(save_dir, f"{base_name}_class_{i}.png"))


# 处理测试图像
image_dir = './test_image'
save_dir = './output_images'
os.makedirs(save_dir, exist_ok=True)

for name in os.listdir(image_dir):
    image_path = os.path.join(image_dir, name)
    inputs, original_image = load_and_preprocess_image(image_path)
    # 进行推理
    with torch.no_grad():
        outputs = model(inputs)
    # 可视化并保存输出
    visualize_and_save_outputs(original_image, outputs, save_dir, os.path.splitext(name)[0])
    
print("Completed!")

验证图像及其分割结果如图所示。

<img src="./test_img.png" width="150" style="display:inline-block;"> <img src="./class_0.png" width="150" style="display:inline-block;"> <img src="./class_1.png" width="150" style="display:inline-block;"> <img src="./class_2.png" width="150" style="display:inline-block;">

2、实现浸润区域不确定性估计，经过20次随机数据处理，得到基于每个像素点的mean_output, uncertainty.

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt


# 使用MC Dropout仅对浸润区域（像素值为3）进行推理
def mc_dropout_inference_infiltration_only(model, inputs, n_samples=20):
    model.eval()
    model.apply(lambda m: m.train() if isinstance(m, nn.Dropout) else None)

    # 进行多次推测并堆叠结果
    outputs = [model(inputs) for _ in range(n_samples)]
    outputs = torch.stack(outputs)
    
    # 针对类别3（浸润区域）的输出
    infiltration_outputs = outputs[:, :, -1, :, :]
    print(infiltration_outputs.shape)

    # 计算类别3的浸润区域均值和不确定性
    mean_output = infiltration_outputs.mean(dim=0)
    uncertainty = infiltration_outputs.std(dim=0)

    # 进行阈值化
    binary_mean_output = (mean_output > 0.5).float()
    binary_uncertainty = (uncertainty > 0.5).float()

    # 转换为 NumPy 数组保存可视化结果
    mean_output_np = binary_mean_output[0].detach().cpu().numpy()  # 维度 (240, 240)
    plt.imshow(mean_output_np, cmap="gray")
    plt.colorbar()
    plt.title("Binary Mean Infiltration Probability")
    plt.savefig("binary_mean_infiltration_probability.png")

    return mean_output, uncertainty


# 定义数据转换
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

in_channels = 1
out_channels = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(in_channels, out_channels, p_dropout=0.5).to(device)
model.load_state_dict(torch.load("./model_checkpoints/unet_model_epoch_25.pth", map_location=device))

image_dir = './test_image'
for name in os.listdir(image_dir):
    image_path = os.path.join(image_dir, name)
    image = Image.open(image_path).convert("L")
    inputs = transform(image).unsqueeze(0).to(device)
    mean_output, uncertainty = mc_dropout_inference_infiltration_only(model, inputs, n_samples=20)

3、基于图像处理得到的mean_output和uncertainty，进行可视化可以得到热力图下的均值图、标准差图。为了更直观的认识浸润区域的不确定性估计，上述代码进行了阈值化处理，二值化均值图像如下图所示。

<img src="./heatmap.png" width="450">